In [1]:
import pandas as pd
import numpy as np
from datetime import date,datetime
import warnings
import copy
from rapidfuzz import process, fuzz


warnings.filterwarnings('ignore')


In [2]:
def check_all_code_columns(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df_columns_lower if "code" in column]

    return matching_columns


def check_all_characters_present(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df.columns if clean_string(column) in columns_to_check_lower]

    return matching_columns

def clean_string(s):
    return s.replace(':', '').replace('-', '').replace(')', '').replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()



In [3]:
today_date = date.today()
today_date=''.join(str(today_date).split('-'))
project_name='SEATTLE'


df=pd.read_csv('seattle_rail2024o2o_export_odbc.csv')

# Details File O2O_STOPS Sheet
detail_df=pd.read_excel('details_project_od_Seattle2 (1).xlsx',sheet_name='O2O_STOPS')

# Details File Stops Sheet
seq_df=pd.read_excel('details_project_od_Seattle2 (1).xlsx',sheet_name='STOPS')

In [4]:
# df=df.iloc[:15,:]
# df.head()

In [5]:
df.shape

(1161, 36)

In [6]:
df.columns

Index(['id', 'Completed', 'Last_page', 'Start_language', 'Date_started',
       'Date_last_action', 'IP_address', 'Referring_URL', 'TIME_ADJUST',
       'INTERV_INIT', 'ORIGRESPID', 'ROUTE_SURVEYED_Code_', 'ROUTE_SURVEYED',
       'ROUTE_SURVEYED_Other_', 'STLINE2_ON_Code_', 'STLINE2_ON',
       'STLINE2_OFF_Code_', 'STLINE2_OFF', 'SSSL_ON_Code_', 'SSSL_ON',
       'SSSL_OFF_Code_', 'SSSL_OFF', 'S1L_ON_Code_', 'S1L_ON', 'S1L_OFF_Code_',
       'S1L_OFF', 'STTAC_ON_Code_', 'STTAC_ON', 'STTAC_OFF_Code_', 'STTAC_OFF',
       'TIME_BOARDED_Code_', 'TIME_BOARDED', 'LANGUAGE_Code_', 'LANGUAGE',
       'LANGUAGE_Other_', 'RESTART_URL'],
      dtype='object')

In [7]:
df['ROUTE_SURVEYED_Code_'].unique()

array(['STLINE2', 'SSSL'], dtype=object)

In [8]:

code_columns=check_all_code_columns(df,df.columns)

code_columns=check_all_characters_present(df,code_columns)
columns_to_remove=[]
for column in code_columns:
    cleaned_column=clean_string(column)
#     if 'routesurveyed' in cleaned_column or 'language' in cleaned_column:
    if 'time' in cleaned_column or 'language' in cleaned_column:
        columns_to_remove.append(column)
columns_to_remove

['TIME_BOARDED_Code_', 'LANGUAGE_Code_']

In [9]:
detail_df.ETC_ROUTE_ID.unique()

array(['SND_1_SLine_00', 'SND_1_1Line_01', 'SND_1_TLine_00',
       'SND_1_2Line_00'], dtype=object)

In [10]:
seq_df.ETC_ROUTE_ID.unique()

array(['CMT_1_101_00', 'CMT_1_101_01', 'CMT_1_102_00', 'CMT_1_102_01',
       'CMT_1_105_00', 'CMT_1_105_01', 'CMT_1_106_00', 'CMT_1_106_01',
       'CMT_1_107_00', 'CMT_1_107_01', 'CMT_1_109_00', 'CMT_1_109_01',
       'CMT_1_111_00', 'CMT_1_111_01', 'CMT_1_112_00', 'CMT_1_112_01',
       'CMT_1_113_00', 'CMT_1_113_01', 'CMT_1_114_00', 'CMT_1_114_01',
       'CMT_1_115_00', 'CMT_1_115_01', 'CMT_1_116_00', 'CMT_1_116_01',
       'CMT_1_119_00', 'CMT_1_119_01', 'CMT_1_120_00', 'CMT_1_120_01',
       'CMT_1_130_00', 'CMT_1_130_01', 'CMT_1_166_00', 'CMT_1_166_01',
       'CMT_1_196_00', 'CMT_1_196_01', 'CMT_1_201_00', 'CMT_1_201_01',
       'CMT_1_202_00', 'CMT_1_202_01', 'CMT_1_209_00', 'CMT_1_209_01',
       'CMT_1_220_00', 'CMT_1_220_01', 'CMT_1_222_00', 'CMT_1_222_01',
       'CMT_1_227_00', 'CMT_1_227_01', 'CMT_1_230_00', 'CMT_1_230_01',
       'CMT_1_240_00', 'CMT_1_240_01', 'CMT_1_247_00', 'CMT_1_247_01',
       'CMT_1_270_00', 'CMT_1_270_01', 'CMT_1_271_00', 'CMT_1_271_01',
      

In [11]:
o2o_df=copy.deepcopy(df)

new_df=o2o_df.drop(columns=columns_to_remove)

code_columns_check=check_all_characters_present(new_df,code_columns)

new_code_columns=check_all_characters_present(new_df,code_columns_check)

route_surveyed_column_check=['routesurveyed']
route_surveyed_column=check_all_characters_present(new_df,route_surveyed_column_check)
route_surveyed_column

['ROUTE_SURVEYED']

In [12]:
new_code_columns

['ROUTE_SURVEYED_Code_',
 'STLINE2_ON_Code_',
 'STLINE2_OFF_Code_',
 'SSSL_ON_Code_',
 'SSSL_OFF_Code_',
 'S1L_ON_Code_',
 'S1L_OFF_Code_',
 'STTAC_ON_Code_',
 'STTAC_OFF_Code_']

In [13]:
new_code_columns

['ROUTE_SURVEYED_Code_',
 'STLINE2_ON_Code_',
 'STLINE2_OFF_Code_',
 'SSSL_ON_Code_',
 'SSSL_OFF_Code_',
 'S1L_ON_Code_',
 'S1L_OFF_Code_',
 'STTAC_ON_Code_',
 'STTAC_OFF_Code_']

In [14]:

def get_route_line(x):
#     return x[1]
    b_values=x.split('(')
    print(b_values)
    if len(b_values)>1:
        print('ENtering in b_values_condition')
        join_value=" ".join(b_values[:-1]).strip()
        values=str(join_value).split(' ')
        return values[-2]
    else:
        print('Entering else block')
        values=x.split(' ')
        print(values)
        if len(values)>1:
            return values[-2]
        return x[1]
    
new_df['Line_Value']=new_df[route_surveyed_column[0]].apply(get_route_line)


['ST 2 Line']
Entering else block
['ST', '2', 'Line']
['ST SOUNDER S LINE']
Entering else block
['ST', 'SOUNDER', 'S', 'LINE']
['ST 2 Line']
Entering else block
['ST', '2', 'Line']
['ST 2 Line']
Entering else block
['ST', '2', 'Line']
['ST 2 Line']
Entering else block
['ST', '2', 'Line']
['ST 2 Line']
Entering else block
['ST', '2', 'Line']
['ST 2 Line']
Entering else block
['ST', '2', 'Line']
['ST 2 Line']
Entering else block
['ST', '2', 'Line']
['ST 2 Line']
Entering else block
['ST', '2', 'Line']
['ST 2 Line']
Entering else block
['ST', '2', 'Line']
['ST 2 Line']
Entering else block
['ST', '2', 'Line']
['ST 2 Line']
Entering else block
['ST', '2', 'Line']
['ST 2 Line']
Entering else block
['ST', '2', 'Line']
['ST 2 Line']
Entering else block
['ST', '2', 'Line']
['ST 2 Line']
Entering else block
['ST', '2', 'Line']
['ST 2 Line']
Entering else block
['ST', '2', 'Line']
['ST 2 Line']
Entering else block
['ST', '2', 'Line']
['ST 2 Line']
Entering else block
['ST', '2', 'Line']
['ST 2 Li

In [15]:
a='STLINE2'

In [16]:
new_df['Line_Value'].unique()

array(['2', 'S'], dtype=object)

In [17]:
a='STLINE2'
get_route_line(a)

['STLINE2']
Entering else block
['STLINE2']


'T'

In [18]:
new_df[['Line_Value']].head()

,Line_Value
0,2
1,S
2,2
3,2
4,2


In [19]:
new_df['Line_Value'].unique()

array(['2', 'S'], dtype=object)

In [20]:

def get_detail_route_line(x):
    values=str(x).split('_')
    if len(values)>1:
        return values[-2][0]
    else:
        return x
    
seq_df['Line_Value']=seq_df['ETC_ROUTE_ID'].apply(get_detail_route_line)


In [21]:
a='SND_1_TLine_00'
get_detail_route_line(a)

'T'

In [22]:
seq_df['Line_Value'].unique()

array(['1', '2', '4', '8', 'S', 'F', '3', '6', '7', '9', '5', 'A', 'B',
       'K', 'N', 'C', 'D', 'E', 'H', 'T', 'L', 'M', 'O', 'P', 'V'],
      dtype=object)

In [23]:
code_columns[1:-2]

['STLINE2_ON_Code_',
 'STLINE2_OFF_Code_',
 'SSSL_ON_Code_',
 'SSSL_OFF_Code_',
 'S1L_ON_Code_',
 'S1L_OFF_Code_',
 'STTAC_ON_Code_',
 'STTAC_OFF_Code_']

In [24]:
new_code_columns

['ROUTE_SURVEYED_Code_',
 'STLINE2_ON_Code_',
 'STLINE2_OFF_Code_',
 'SSSL_ON_Code_',
 'SSSL_OFF_Code_',
 'S1L_ON_Code_',
 'S1L_OFF_Code_',
 'STTAC_ON_Code_',
 'STTAC_OFF_Code_']

In [25]:
required_code_columns=code_columns[:-2]
required_code_columns

['ROUTE_SURVEYED_Code_',
 'STLINE2_ON_Code_',
 'STLINE2_OFF_Code_',
 'SSSL_ON_Code_',
 'SSSL_OFF_Code_',
 'S1L_ON_Code_',
 'S1L_OFF_Code_',
 'STTAC_ON_Code_',
 'STTAC_OFF_Code_']

In [26]:

for _, row in new_df.iterrows():
    counter = 0
    for i in range(1, len(new_code_columns)):
        seq_value = row[code_columns[i]]
        value_column_check = [''.join(part for part in required_code_columns[i].split('_') if part.lower() != 'code').lower()]
        value_column = check_all_characters_present(new_df, value_column_check)
        value = row[value_column[0]]

        if pd.isnull(value) or pd.isnull(seq_value):
            continue  # Skip if value or seq_value is null

        stop_value = value

        # First, filter sequence_value using the original logic
        sequence_value = seq_df[
            (seq_df['Line_Value'] == row['Line_Value']) & 
            (seq_df['ETC_STOP_NAME'].str.lower().str.contains(value.lower()))
        ]

        # If sequence_value is empty, apply fuzzy matching
        if sequence_value.empty:
            # Use fuzzy matching to find the best match for ETC_STOP_NAME
            best_match, score, _ = process.extractOne(
                value.lower(),
                seq_df[seq_df['Line_Value'] == row['Line_Value']]['ETC_STOP_NAME'].str.lower(),
                scorer=fuzz.ratio
            )
            print(f'{best_match},{value},{score},{row["Line_Value"]}')
            # print(score,best_match,value.lower())
            # print()
            if score >= 40:
                sequence_value = seq_df[
                    (seq_df['Line_Value'] == row['Line_Value']) & 
                    (seq_df['ETC_STOP_NAME'].str.lower() == best_match)
                ]
                
        # Proceed only if we have a valid sequence_value after both methods
        if not sequence_value.empty:
            seq_values_array = sequence_value['seq_fixed'].values

            if seq_values_array.size > 0:
                differences = np.abs(seq_values_array - seq_value)
                nearest_index = np.argmin(differences)
                nearest_value = seq_values_array[nearest_index]
                matched_value = sequence_value[sequence_value['seq_fixed'] == nearest_value]

                if not matched_value.empty:
                    counter += 1
                    if counter == 1:
                        new_df.loc[row.name, ['ROUTE_DESCRIPTION[CODE]', 'ETC_ROUTE_NAME', 'BOARD_ID', 'BOARD_STOP[CODE]', 'BOARD_STOP_NAME']] = [
                            matched_value.iloc[0]['ETC_ROUTE_ID'],
                            matched_value.iloc[0]['ETC_ROUTE_NAME'],
                            matched_value.iloc[0]['seq_fixed'],
                            matched_value.iloc[0]['ETC_STOP_ID'],
                            matched_value.iloc[0]['ETC_STOP_NAME'],
                        ]
                    else:  # counter > 1
                        new_df.loc[row.name, ['ALIGHT_ID', 'ALIGHT_STOP[CODE]', 'ALIGHT_STOP_NAME']] = [
                            matched_value.iloc[0]['seq_fixed'],
                            matched_value.iloc[0]['ETC_STOP_ID'],
                            matched_value.iloc[0]['ETC_STOP_NAME'],
                        ]
            else:
                print(f"No valid seq_fixed values found for {row['Line_Value']} and {value}.")


In [27]:

new_df=pd.merge(new_df,df[['id','TIME_BOARDED_Code_']],on='id',how='left')

new_df['Date'] = np.where(new_df['Completed'].notna(), new_df['Completed'], new_df['Date_started'])


In [28]:
def get_day_name(x):
    if pd.isna(x):
        return np.nan
    x = str(x)
    try:
        date_object = datetime.strptime(x, '%Y-%m-%d %H:%M:%S')
    except ValueError:
        try:
            date_object = datetime.strptime(x, '%d/%m/%Y %H:%M')
        except ValueError:
            return np.nan
    day_name = date_object.strftime('%A')
    if day_name in ['Saturday', 'Sunday']:
        return 'Weekend'
    return 'Weekday'

new_df['Day']=new_df['Date'].apply(get_day_name)

new_df=new_df[new_df['INTERV_INIT']!='999']


In [29]:

# time_period_mapping = {
#     'PM1': 'PM',
#     'PM2': 'Evening',
#     'MID': 'MIDDAY',
#     'AM': 'AM'
# }

# time_period_code_mapping = {
#     'PM':2,
#     'Evening':4,
#     'MIDDAY':1,
#     'AM': 0
# }

# new_df['TIME PERIOD'] = new_df['TIME_BOARDED_Code_'].map(time_period_mapping).fillna(new_df['TIME_BOARDED_Code_'])
# new_df['TIME PERIOD[Code]'] = new_df['TIME PERIOD'].map(time_period_code_mapping)


In [30]:

for _, row in new_df.iterrows():
    # Extract the alight sequence
    alight_seq_df = seq_df[(seq_df['ETC_ROUTE_NAME'] == row['ETC_ROUTE_NAME']) & 
                           (seq_df['ETC_STOP_ID'] == row['ALIGHT_STOP[CODE]'])][['seq_fixed']]
    # alight_seq_df = seq_df[(seq_df['ETC_ROUTE_NAME'] == row['ETC_ROUTE_NAME']) & 
    #                     (seq_df['ETC_STOP_NAME'] == row['ALIGHT_STOP_NAME'])][['seq_fixed']]
    # alight_seq_lat_long_df = seq_df[(seq_df['ETC_STOP_NAME'] == row['ALIGHT_STOP_NAME']) & 
    #                                 (seq_df['ETC_STOP_ID'] == row['ALIGHT_STOP[CODE]'])][['stop_lat6', 'stop_lon6']]
    alight_seq_lat_long_df = seq_df[(seq_df['ETC_ROUTE_NAME'] == row['ETC_ROUTE_NAME']) & 
                            (seq_df['ETC_STOP_NAME'] == row['ALIGHT_STOP_NAME'])][['stop_lat6', 'stop_lon6']]
    if not alight_seq_df.empty:
        alight_seq = alight_seq_df.iloc[0, 0]
    else:
        alight_seq = 0

    # Extract the board sequence
    board_seq_df = seq_df[(seq_df['ETC_ROUTE_NAME'] == row['ETC_ROUTE_NAME']) & 
                          (seq_df['ETC_STOP_ID'] == row['BOARD_STOP[CODE]'])][['seq_fixed']]
    # board_seq_df = seq_df[(seq_df['ETC_ROUTE_NAME'] == row['ETC_ROUTE_NAME']) & 
    #                     (seq_df['ETC_STOP_NAME'] == row['BOARD_STOP_NAME'])][['seq_fixed']]
    # board_seq_lat_long_df = seq_df[(seq_df['ETC_STOP_NAME'] == row['BOARD_STOP_NAME']) & 
    #                             (seq_df['ETC_STOP_ID'] == row['BOARD_STOP[CODE]'])][['stop_lat6', 'stop_lon6']]
    board_seq_lat_long_df = seq_df[(seq_df['ETC_ROUTE_NAME'] == row['ETC_ROUTE_NAME']) & 
                                   (seq_df['ETC_STOP_NAME'] == row['BOARD_STOP_NAME'])][['stop_lat6', 'stop_lon6']]
    
    if not board_seq_df.empty:
        board_seq = board_seq_df.iloc[0, 0]
    else:
        board_seq = 0

    # Assign sequence values to new_df
    new_df.loc[row.name, 'Alight_Seq'] = alight_seq
    new_df.loc[row.name, 'Board_Seq'] = board_seq

    # Assign latitude and longitude values to new_df
    if not board_seq_lat_long_df.empty:
        new_df.loc[row.name, 'BOARD_STOP_ON_LAT'] = board_seq_lat_long_df.iloc[0, 0]
        new_df.loc[row.name, 'BOARD_STOP_ON_LONG'] = board_seq_lat_long_df.iloc[0, 1]
    else:
        new_df.loc[row.name, 'BOARD_STOP_ON_LAT'] = None
        new_df.loc[row.name, 'BOARD_STOP_ON_LONG'] = None

    if not alight_seq_lat_long_df.empty:
        new_df.loc[row.name, 'ALIGHT_STOP_ON_LAT'] = alight_seq_lat_long_df.iloc[0, 0]
        new_df.loc[row.name, 'ALIGHT_STOP_ON_LONG'] = alight_seq_lat_long_df.iloc[0, 1]
    else:
        new_df.loc[row.name, 'ALIGHT_STOP_ON_LAT'] = None
        new_df.loc[row.name, 'ALIGHT_STOP_ON_LONG'] = None

new_df['SEQ_CHECK'] = new_df['Alight_Seq'] - new_df['Board_Seq']



In [31]:
new_df = new_df[new_df['SEQ_CHECK'] != 0].copy()

ids_list = []

for _, row in new_df.iterrows():
    if row['SEQ_CHECK'] < 0:
        # Extract route code and modify it
        ids_list.append(row['id'])

        route_code = row['ROUTE_DESCRIPTION[CODE]']
        new_route_code = f"{'_'.join(route_code.split('_')[:-1])}_01" if route_code.split('_')[-1] == '00' else f"{'_'.join(route_code.split('_')[:-1])}_00"
        
        # Get new route name
        new_route_name_row = seq_df[seq_df['ETC_ROUTE_ID'] == new_route_code]
        if not new_route_name_row.empty:
            new_route_name = new_route_name_row['ETC_ROUTE_NAME'].iloc[0]
        else:
            # Handle the case when the new route name is not found
            continue  # Skip to the next iteration if new route code is not valid

        # Board stop details
        board_stop_row = seq_df[(seq_df['ETC_ROUTE_ID'] == new_route_code) & (seq_df['ETC_STOP_NAME'] == row['BOARD_STOP_NAME'])]
        if not board_stop_row.empty:
            board_stop_code = board_stop_row['ETC_STOP_ID'].iloc[0]
            board_stop_lat_lon = board_stop_row[['seq_fixed', 'stop_lat6', 'stop_lon6']].iloc[0]
        else:
            # Skip if no board stop information is found
            continue

        # Alight stop details
        alight_stop_row = seq_df[(seq_df['ETC_ROUTE_ID'] == new_route_code) & (seq_df['ETC_STOP_NAME'] == row['ALIGHT_STOP_NAME'])]
        if not alight_stop_row.empty:
            alight_stop_code = alight_stop_row['ETC_STOP_ID'].iloc[0]
            alight_stop_lat_lon = alight_stop_row[['seq_fixed', 'stop_lat6', 'stop_lon6']].iloc[0]
        else:
            # Skip if no alight stop information is found
            continue

        # Update new_df with the retrieved values
        new_df.loc[row.name, 'ROUTE_DESCRIPTION[CODE]'] = new_route_code
        new_df.loc[row.name, 'ETC_ROUTE_NAME'] = new_route_name

        new_df.loc[row.name, 'Board_Seq'] = board_stop_lat_lon['seq_fixed']
        new_df.loc[row.name, 'BOARD_STOP[CODE]'] = board_stop_code

        new_df.loc[row.name, 'BOARD_STOP_ON_LAT'] = board_stop_lat_lon['stop_lat6']
        new_df.loc[row.name, 'BOARD_STOP_ON_LONG'] = board_stop_lat_lon['stop_lon6']

        new_df.loc[row.name, 'Alight_Seq'] = alight_stop_lat_lon['seq_fixed']
        new_df.loc[row.name, 'ALIGHT_STOP[CODE]'] = alight_stop_code

        new_df.loc[row.name, 'ALIGHT_STOP_ON_LAT'] = alight_stop_lat_lon['stop_lat6']
        new_df.loc[row.name, 'ALIGHT_STOP_ON_LONG'] = alight_stop_lat_lon['stop_lon6']


In [32]:
final_df=new_df[['id','Date','Day','TIME_BOARDED','ROUTE_DESCRIPTION[CODE]','ETC_ROUTE_NAME','Line_Value','BOARD_STOP[CODE]','Board_Seq' ,'BOARD_STOP_NAME','BOARD_STOP_ON_LAT','BOARD_STOP_ON_LONG','ALIGHT_STOP[CODE]','Alight_Seq' ,'ALIGHT_STOP_NAME','ALIGHT_STOP_ON_LAT','ALIGHT_STOP_ON_LONG']]


In [33]:
# final_df.to_csv('SEATTLE_NEW_O2O.csv',index=False)

In [34]:
final_df.head()

,id,Date,Day,TIME_BOARDED,ROUTE_DESCRIPTION[CODE],ETC_ROUTE_NAME,Line_Value,BOARD_STOP[CODE],Board_Seq,BOARD_STOP_NAME,BOARD_STOP_ON_LAT,BOARD_STOP_ON_LONG,ALIGHT_STOP[CODE],Alight_Seq,ALIGHT_STOP_NAME,ALIGHT_STOP_ON_LAT,ALIGHT_STOP_ON_LONG
2,105546,2024-10-07 08:05:52,Weekday,AM Peak (Before 9:00am),MET_1_249_00,MET_1_249_00,2,MET_1_249_00_84264,1.0,South Bellevue Station Bus Plaza - Bay 2,47.586960,-122.190590,SND_1_2Line_00_E15-T2,0.0,Bellevue Downtown,None,None
3,105547,2024-10-07 08:08:37,Weekday,AM Peak (Before 9:00am),MET_1_249_00,MET_1_249_00,2,MET_1_249_00_84264,1.0,South Bellevue Station Bus Plaza - Bay 2,47.586960,-122.190590,SND_1_2Line_00_E15-T2,0.0,Bellevue Downtown,None,None
4,105548,2024-10-07 08:08:48,Weekday,AM Peak (Before 9:00am),MET_1_249_00,MET_1_249_00,2,MET_1_249_00_84264,1.0,South Bellevue Station Bus Plaza - Bay 2,47.586960,-122.190590,SND_1_2Line_00_E15-T2,0.0,Bellevue Downtown,None,None
5,105549,2024-10-07 08:09:10,Weekday,AM Peak (Before 9:00am),SND_1_2Line_00,(Sound) 2 Line South Bellevue - Redmond Tech [...,2,SND_1_2Line_00_E15-T2,3.0,Bellevue Downtown,47.615183,-122.191303,SND_1_2Line_00_E21-T2,5.0,Spring District,47.623776,-122.178239
6,105550,2024-10-07 08:09:25,Weekday,AM Peak (Before 9:00am),SND_1_2Line_00,(Sound) 2 Line South Bellevue - Redmond Tech [...,2,SND_1_2Line_00_E15-T2,3.0,Bellevue Downtown,47.615183,-122.191303,SND_1_2Line_00_E21-T2,5.0,Spring District,47.623776,-122.178239
